# 02 — Training

This notebook covers the full training pipeline for the EuroSAT land-cover classifier:

1. **Setup** — imports, config, seed
2. **Data Loading** — stratified split, normalization, DataLoaders
3. **Architecture Comparison** — SimpleCNN vs ResNetSmall
4. **Loss Function Comparison** — CrossEntropy vs Label Smoothing
5. **Learning Rate Scheduler Comparison** — StepLR vs CosineAnnealingLR
6. **Overfitting Experiment** — cause and correction
7. **Underfitting Experiment** — cause and correction
8. **Hyperparameter Tuning** — grid search over LR, batch size, weight decay
9. **Final Training** — best config with early stopping, save weights

All heavy logic lives in `src/` modules — this notebook orchestrates, visualizes, and narrates.

## 0. Environment Setup (Colab / Local)

Run this cell first. It auto-detects Colab vs local Jupyter and sets up the environment.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    REPO_URL = 'https://github.com/sabyasachi1992/eurosat-land-cover-ood.git'
    REPO_DIR = '/content/eurosat-land-cover-ood'
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
    if not os.path.isdir('EuroSAT/2750'):
        print('Downloading EuroSAT RGB dataset...')
        !wget --no-check-certificate -q https://madm.dfki.de/files/sentinel/EuroSAT.zip -O EuroSAT.zip
        import zipfile
        if not zipfile.is_zipfile('EuroSAT.zip'):
            print('DFKI mirror failed, trying Kaggle mirror...')
            !rm -f EuroSAT.zip
            !pip install -q kagglehub
            import kagglehub, glob
            path = kagglehub.dataset_download('apollo2506/eurosat-dataset')
            !mkdir -p EuroSAT
            candidates = glob.glob(f'{path}/**/2750', recursive=True)
            if candidates:
                !cp -r {candidates[0]} EuroSAT/2750
            else:
                !cp -r {path}/* EuroSAT/
        else:
            !unzip -q EuroSAT.zip
            !mkdir -p EuroSAT
            if os.path.isdir('2750'):
                !mv 2750 EuroSAT/2750
            elif os.path.isdir('EuroSAT_RGB'):
                !mv EuroSAT_RGB EuroSAT/2750
            !rm -f EuroSAT.zip
        print(f'Done. Classes: {sorted(os.listdir("EuroSAT/2750"))}')
    PROJECT_ROOT = REPO_DIR
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        PROJECT_ROOT = os.path.abspath('..')
    elif os.path.isfile('config.yaml'):
        PROJECT_ROOT = os.getcwd()
    else:
        PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
assert os.path.isfile('config.yaml'), f'config.yaml not found in {os.getcwd()}'

## 1. Setup

In [ ]:
%matplotlib inline

import copy
import json
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from src.config import Config
from src.data.dataset import discover_images, stratified_split, EuroSATDataset
from src.data.transforms import load_norm_stats, get_train_transform, get_eval_transform
from src.models.factory import create_model
from src.models.simple_cnn import SimpleCNN
from src.models.resnet_small import ResNetSmall
from src.training.trainer import Trainer
from src.utils.seed import set_global_seed
from src.utils.visualization import plot_training_curves, plot_lr_curve

In [ ]:
config = Config.load('config.yaml')
set_global_seed(config.seed)

dataset_root = config.dataset_root
output_dir = config.output_dir
norm_stats_path = config.norm_stats_path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Seed:   {config.seed}')

## 2. Data Loading

We discover known-class images, perform the same stratified split as notebook 01,
load the pre-computed normalization statistics, and build train/val DataLoaders.

- **Training** loader uses augmentation (flips, rotation, color jitter) + normalization.
- **Validation** loader uses normalization only — no augmentation.

Augmentation justification:
- **Horizontal/vertical flip**: satellite patches have no canonical orientation, so flips are semantically valid.
- **Random rotation (±15°)**: slight rotational invariance helps generalize across sensor angles.
- **Color jitter**: simulates atmospheric and seasonal variation in Sentinel-2 imagery.

In [ ]:
# Discover and split
known_paths, known_labels = discover_images(dataset_root, config.known_classes)
train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = stratified_split(
    known_paths, known_labels,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    test_ratio=config.test_ratio,
    seed=config.seed,
)

print(f'Train: {len(train_paths):,} images')
print(f'Val:   {len(val_paths):,} images')
print(f'Test:  {len(test_paths):,} images  (held out for notebook 03)')

In [ ]:
# Load normalization stats computed in notebook 01
mean, std = load_norm_stats(norm_stats_path)
print(f'Norm mean: {[round(m, 4) for m in mean]}')
print(f'Norm std:  {[round(s, 4) for s in std]}')

In [ ]:
def build_loaders(train_paths, train_labels, val_paths, val_labels, cfg, mean, std):
    """Build train and val DataLoaders with appropriate transforms."""
    train_transform = get_train_transform(cfg, mean, std)
    eval_transform = get_eval_transform(mean, std)

    train_ds = EuroSATDataset(train_paths, train_labels, transform=train_transform)
    val_ds = EuroSATDataset(val_paths, val_labels, transform=eval_transform)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader

train_loader, val_loader = build_loaders(
    train_paths, train_labels, val_paths, val_labels, config, mean, std
)

print(f'Train batches: {len(train_loader)} (batch_size={config.batch_size})')
print(f'Val batches:   {len(val_loader)}')

# Quick shape check
imgs, labels = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}, Labels shape: {labels.shape}')

## 3. Architecture Comparison — SimpleCNN vs ResNetSmall

We compare two architectures trained from scratch:

| Architecture | Approx. Params | Depth | Key Feature |
|---|---|---|---|
| SimpleCNN | ~50K–100K | 3 conv blocks | Shallow baseline |
| ResNetSmall | ~500K–2M | 4 residual stages | Skip connections |

Both are trained for 20 epochs with the same hyperparameters to isolate the effect of architecture.

In [ ]:
COMPARISON_EPOCHS = 20

def make_config_variant(**overrides):
    """Create a Config copy with in-memory overrides (no file modification)."""
    cfg = copy.deepcopy(config)
    for k, v in overrides.items():
        setattr(cfg, k, v)
    return cfg

def count_params(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
# --- SimpleCNN ---
set_global_seed(config.seed)
cfg_cnn = make_config_variant(
    architecture='simple_cnn',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,  # disable early stopping for comparison
    weights_path=os.path.join(output_dir, 'cnn_compare.pt'),
    scheduler='cosine_annealing',
    scheduler_params={'T_max': COMPARISON_EPOCHS},
)
tl, vl = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_cnn, mean, std)
model_cnn = create_model(cfg_cnn)
print(f'SimpleCNN params: {count_params(model_cnn):,}')

trainer_cnn = Trainer(model_cnn, tl, vl, cfg_cnn)
history_cnn = trainer_cnn.train()
print(f'SimpleCNN best val acc: {max(history_cnn["val_acc"]):.4f}')

In [ ]:
# --- ResNetSmall ---
set_global_seed(config.seed)
cfg_resnet = make_config_variant(
    architecture='resnet_small',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,
    weights_path=os.path.join(output_dir, 'resnet_compare.pt'),
    scheduler='cosine_annealing',
    scheduler_params={'T_max': COMPARISON_EPOCHS},
)
tl, vl = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_resnet, mean, std)
model_resnet = create_model(cfg_resnet)
print(f'ResNetSmall params: {count_params(model_resnet):,}')

trainer_resnet = Trainer(model_resnet, tl, vl, cfg_resnet)
history_resnet = trainer_resnet.train()
print(f'ResNetSmall best val acc: {max(history_resnet["val_acc"]):.4f}')

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, COMPARISON_EPOCHS + 1)

ax1.plot(epochs_range, history_cnn['train_loss'], '--', label='SimpleCNN train')
ax1.plot(epochs_range, history_cnn['val_loss'], '-', label='SimpleCNN val')
ax1.plot(epochs_range, history_resnet['train_loss'], '--', label='ResNetSmall train')
ax1.plot(epochs_range, history_resnet['val_loss'], '-', label='ResNetSmall val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss Curves')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history_cnn['train_acc'], '--', label='SimpleCNN train')
ax2.plot(epochs_range, history_cnn['val_acc'], '-', label='SimpleCNN val')
ax2.plot(epochs_range, history_resnet['train_acc'], '--', label='ResNetSmall train')
ax2.plot(epochs_range, history_resnet['val_acc'], '-', label='ResNetSmall val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy Curves')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSimpleCNN  — params: {count_params(model_cnn):>8,}  best val acc: {max(history_cnn["val_acc"]):.4f}')
print(f'ResNetSmall — params: {count_params(model_resnet):>8,}  best val acc: {max(history_resnet["val_acc"]):.4f}')

### Architecture Analysis

**ResNetSmall outperforms SimpleCNN** for several reasons:

1. **Greater capacity**: ~10–20× more parameters allow the model to learn richer feature representations for the 6 land-cover classes.
2. **Depth with residual connections**: The skip connections in ResNetSmall enable training of deeper networks without vanishing gradients. SimpleCNN's 3 blocks limit its representational depth.
3. **Hierarchical features**: The 4-stage residual architecture (32→64→128→192 channels) builds increasingly abstract features, which is critical for distinguishing visually similar classes like AnnualCrop vs HerbaceousVegetation.

SimpleCNN serves as a useful baseline to quantify the benefit of architectural depth and residual learning.

**Conclusion**: We select **ResNetSmall** as the primary architecture for all subsequent experiments.

## 4. Loss Function Comparison

We compare two loss functions using ResNetSmall for 20 epochs:

- **CrossEntropyLoss**: Standard log-loss, assigns full probability to the ground-truth class.
- **Label Smoothing CrossEntropyLoss** (smoothing=0.1): Distributes 10% of the probability mass uniformly across all classes, preventing the model from becoming overconfident.

Label smoothing acts as a regularizer — it penalizes extreme logit values and can improve calibration.

In [ ]:
# --- CrossEntropyLoss ---
set_global_seed(config.seed)
cfg_ce = make_config_variant(
    architecture='resnet_small',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,
    loss_function='cross_entropy',
    weights_path=os.path.join(output_dir, 'ce_compare.pt'),
    scheduler='cosine_annealing',
    scheduler_params={'T_max': COMPARISON_EPOCHS},
)
tl, vl = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_ce, mean, std)
trainer_ce = Trainer(create_model(cfg_ce), tl, vl, cfg_ce)
history_ce = trainer_ce.train()
print(f'CrossEntropy best val acc: {max(history_ce["val_acc"]):.4f}')

In [ ]:
# --- Label Smoothing ---
set_global_seed(config.seed)
cfg_ls = make_config_variant(
    architecture='resnet_small',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,
    loss_function='label_smoothing',
    label_smoothing=0.1,
    weights_path=os.path.join(output_dir, 'ls_compare.pt'),
    scheduler='cosine_annealing',
    scheduler_params={'T_max': COMPARISON_EPOCHS},
)
tl, vl = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_ls, mean, std)
trainer_ls = Trainer(create_model(cfg_ls), tl, vl, cfg_ls)
history_ls = trainer_ls.train()
print(f'Label Smoothing best val acc: {max(history_ls["val_acc"]):.4f}')

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, COMPARISON_EPOCHS + 1)

ax1.plot(epochs_range, history_ce['val_loss'], '-o', markersize=3, label='CrossEntropy')
ax1.plot(epochs_range, history_ls['val_loss'], '-s', markersize=3, label='Label Smoothing')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Val Loss'); ax1.set_title('Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history_ce['val_acc'], '-o', markersize=3, label='CrossEntropy')
ax2.plot(epochs_range, history_ls['val_acc'], '-s', markersize=3, label='Label Smoothing')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Accuracy'); ax2.set_title('Validation Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'CrossEntropy    — best val acc: {max(history_ce["val_acc"]):.4f}')
print(f'Label Smoothing — best val acc: {max(history_ls["val_acc"]):.4f}')

### Loss Function Analysis

**Key observations:**

- **Validation accuracy** is comparable between the two losses, with label smoothing often showing a slight edge in generalization.
- **Label smoothing produces higher validation loss values** — this is expected because the loss target is no longer a one-hot vector, so the minimum achievable loss is higher by design.
- **Confidence calibration**: Label smoothing prevents the model from assigning near-1.0 softmax probabilities to predictions. This is beneficial for OOD detection, where overconfident predictions on unknown classes are problematic.
- **Generalization**: By discouraging the model from memorizing exact class boundaries, label smoothing acts as an implicit regularizer.

**Conclusion**: We select **Label Smoothing (0.1)** for the final model because better-calibrated confidence scores directly benefit our downstream OOD detection pipeline.

## 5. Learning Rate Scheduler Comparison

We compare two LR scheduling strategies using ResNetSmall for 20 epochs:

- **StepLR** (step_size=10, gamma=0.1): Drops the LR by 10× every 10 epochs. Simple but coarse.
- **CosineAnnealingLR** (T_max=20): Smoothly decays the LR following a cosine curve from the initial value to near zero. Provides gradual refinement.

In [ ]:
# --- StepLR ---
set_global_seed(config.seed)
cfg_step = make_config_variant(
    architecture='resnet_small',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,
    scheduler='step_lr',
    scheduler_params={'step_size': 10, 'gamma': 0.1},
    weights_path=os.path.join(output_dir, 'step_compare.pt'),
)
tl, vl = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_step, mean, std)
trainer_step = Trainer(create_model(cfg_step), tl, vl, cfg_step)
history_step = trainer_step.train()
print(f'StepLR best val acc: {max(history_step["val_acc"]):.4f}')

In [ ]:
# --- CosineAnnealingLR ---
set_global_seed(config.seed)
cfg_cosine = make_config_variant(
    architecture='resnet_small',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,
    scheduler='cosine_annealing',
    scheduler_params={'T_max': COMPARISON_EPOCHS},
    weights_path=os.path.join(output_dir, 'cosine_compare.pt'),
)
tl, vl = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_cosine, mean, std)
trainer_cosine = Trainer(create_model(cfg_cosine), tl, vl, cfg_cosine)
history_cosine = trainer_cosine.train()
print(f'CosineAnnealing best val acc: {max(history_cosine["val_acc"]):.4f}')

In [ ]:
# Plot LR schedules and val accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, COMPARISON_EPOCHS + 1)

ax1.plot(epochs_range, history_step['lr'], '-o', markersize=3, label='StepLR')
ax1.plot(epochs_range, history_cosine['lr'], '-s', markersize=3, label='CosineAnnealing')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Learning Rate'); ax1.set_title('LR Schedule')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history_step['val_acc'], '-o', markersize=3, label='StepLR')
ax2.plot(epochs_range, history_cosine['val_acc'], '-s', markersize=3, label='CosineAnnealing')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Accuracy'); ax2.set_title('Validation Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'StepLR          — best val acc: {max(history_step["val_acc"]):.4f}')
print(f'CosineAnnealing — best val acc: {max(history_cosine["val_acc"]):.4f}')

### Scheduler Analysis

**Key observations:**

- **StepLR** maintains a constant high LR for the first 10 epochs, then drops abruptly. This can cause instability early on and wastes fine-tuning potential in the second half.
- **CosineAnnealingLR** provides a smooth, gradual decay that allows the optimizer to explore broadly early and refine precisely later. This typically leads to better convergence.
- The cosine schedule avoids the "shock" of sudden LR drops that can temporarily destabilize training.

**Conclusion**: We select **CosineAnnealingLR** for the final model — its smooth decay profile consistently produces better or equal validation accuracy compared to StepLR.

## 6. Overfitting Experiment

**Goal**: Demonstrate a clear overfitting scenario where training accuracy is high but validation accuracy is substantially lower.

**Setup**: We train ResNetSmall with:
- **No augmentation** (all augmentation toggles disabled)
- **No weight decay** (weight_decay=0)
- **No dropout** (the default ResNetSmall has no dropout, so this is already the case)

Without regularization, the model memorizes the training set rather than learning generalizable features.

In [ ]:
# --- Overfitting: no augmentation, no weight decay ---
set_global_seed(config.seed)
cfg_overfit = make_config_variant(
    architecture='resnet_small',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,
    weight_decay=0.0,
    augmentation={},  # disable all augmentation
    weights_path=os.path.join(output_dir, 'overfit_demo.pt'),
    scheduler='cosine_annealing',
    scheduler_params={'T_max': COMPARISON_EPOCHS},
)
tl_of, vl_of = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_overfit, mean, std)
trainer_overfit = Trainer(create_model(cfg_overfit), tl_of, vl_of, cfg_overfit)
history_overfit = trainer_overfit.train()

print(f'Overfit — final train acc: {history_overfit["train_acc"][-1]:.4f}')
print(f'Overfit — final val acc:   {history_overfit["val_acc"][-1]:.4f}')
print(f'Overfit — gap:             {history_overfit["train_acc"][-1] - history_overfit["val_acc"][-1]:.4f}')

In [ ]:
# --- Correction: add augmentation + weight decay ---
set_global_seed(config.seed)
cfg_overfit_fix = make_config_variant(
    architecture='resnet_small',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,
    weight_decay=1e-4,
    augmentation=config.augmentation,  # restore full augmentation
    weights_path=os.path.join(output_dir, 'overfit_fix.pt'),
    scheduler='cosine_annealing',
    scheduler_params={'T_max': COMPARISON_EPOCHS},
)
tl_fix, vl_fix = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_overfit_fix, mean, std)
trainer_overfit_fix = Trainer(create_model(cfg_overfit_fix), tl_fix, vl_fix, cfg_overfit_fix)
history_overfit_fix = trainer_overfit_fix.train()

print(f'Fixed — final train acc: {history_overfit_fix["train_acc"][-1]:.4f}')
print(f'Fixed — final val acc:   {history_overfit_fix["val_acc"][-1]:.4f}')
print(f'Fixed — gap:             {history_overfit_fix["train_acc"][-1] - history_overfit_fix["val_acc"][-1]:.4f}')

In [ ]:
# Plot overfitting vs corrected
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, COMPARISON_EPOCHS + 1)

ax1.plot(epochs_range, history_overfit['train_acc'], '--r', label='Overfit train')
ax1.plot(epochs_range, history_overfit['val_acc'], '-r', label='Overfit val')
ax1.fill_between(epochs_range, history_overfit['train_acc'], history_overfit['val_acc'], alpha=0.15, color='red')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.set_title('Overfitting: No Aug, No Weight Decay')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history_overfit_fix['train_acc'], '--g', label='Corrected train')
ax2.plot(epochs_range, history_overfit_fix['val_acc'], '-g', label='Corrected val')
ax2.fill_between(epochs_range, history_overfit_fix['train_acc'], history_overfit_fix['val_acc'], alpha=0.15, color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Corrected: Aug + Weight Decay')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Overfitting Analysis

**Cause**: Excessive model capacity (ResNetSmall ~500K+ params) without regularization. With no augmentation and no weight decay, the model memorizes training examples rather than learning generalizable patterns. The training accuracy climbs toward 1.0 while validation accuracy plateaus or degrades.

**Correction**: Adding data augmentation (flips, rotation, jitter) and weight decay (1e-4) closes the train/val gap significantly:
- Augmentation forces the model to be invariant to irrelevant transformations.
- Weight decay penalizes large weights, preventing the model from fitting noise.

The corrected model shows a much smaller train/val accuracy gap while maintaining or improving validation performance.

## 7. Underfitting Experiment

**Goal**: Demonstrate underfitting where both training and validation accuracy are low.

**Setup**: We train SimpleCNN (limited capacity) with:
- **Excessive weight decay** (0.01) — heavily penalizes learned weights
- **Very low learning rate** (0.0001) — slow convergence
- **Few epochs** (10) — insufficient training time

The combination of insufficient capacity and excessive regularization prevents the model from fitting even the training data.

In [ ]:
UNDERFIT_EPOCHS = 10

# --- Underfitting: small model + excessive regularization ---
set_global_seed(config.seed)
cfg_underfit = make_config_variant(
    architecture='simple_cnn',
    epochs=UNDERFIT_EPOCHS,
    early_stopping_patience=UNDERFIT_EPOCHS + 1,
    weight_decay=0.01,
    learning_rate=0.0001,
    weights_path=os.path.join(output_dir, 'underfit_demo.pt'),
    scheduler='cosine_annealing',
    scheduler_params={'T_max': UNDERFIT_EPOCHS},
)
tl_uf, vl_uf = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_underfit, mean, std)
trainer_underfit = Trainer(create_model(cfg_underfit), tl_uf, vl_uf, cfg_underfit)
history_underfit = trainer_underfit.train()

print(f'Underfit — final train acc: {history_underfit["train_acc"][-1]:.4f}')
print(f'Underfit — final val acc:   {history_underfit["val_acc"][-1]:.4f}')

In [ ]:
# --- Correction: ResNetSmall with moderate regularization ---
set_global_seed(config.seed)
cfg_underfit_fix = make_config_variant(
    architecture='resnet_small',
    epochs=COMPARISON_EPOCHS,
    early_stopping_patience=COMPARISON_EPOCHS + 1,
    weight_decay=1e-4,
    learning_rate=0.001,
    weights_path=os.path.join(output_dir, 'underfit_fix.pt'),
    scheduler='cosine_annealing',
    scheduler_params={'T_max': COMPARISON_EPOCHS},
)
tl_fix2, vl_fix2 = build_loaders(train_paths, train_labels, val_paths, val_labels, cfg_underfit_fix, mean, std)
trainer_underfit_fix = Trainer(create_model(cfg_underfit_fix), tl_fix2, vl_fix2, cfg_underfit_fix)
history_underfit_fix = trainer_underfit_fix.train()

print(f'Fixed — final train acc: {history_underfit_fix["train_acc"][-1]:.4f}')
print(f'Fixed — final val acc:   {history_underfit_fix["val_acc"][-1]:.4f}')

In [ ]:
# Plot underfitting vs corrected
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

uf_range = range(1, UNDERFIT_EPOCHS + 1)
ax1.plot(uf_range, history_underfit['train_acc'], '--r', label='Underfit train')
ax1.plot(uf_range, history_underfit['val_acc'], '-r', label='Underfit val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.set_title('Underfitting: SimpleCNN + Heavy Reg + Low LR')
ax1.set_ylim(0, 1); ax1.legend(); ax1.grid(True, alpha=0.3)

fix_range = range(1, COMPARISON_EPOCHS + 1)
ax2.plot(fix_range, history_underfit_fix['train_acc'], '--g', label='Corrected train')
ax2.plot(fix_range, history_underfit_fix['val_acc'], '-g', label='Corrected val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Corrected: ResNetSmall + Moderate Reg')
ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Underfitting Analysis

**Cause**: Insufficient model capacity combined with excessive regularization. SimpleCNN has only ~50K–100K parameters — too few to capture the complexity of 6 land-cover classes. Adding heavy weight decay (0.01) and a very low learning rate (0.0001) with only 10 epochs means the model barely learns anything. Both training and validation accuracy remain low.

**Correction**: Switching to ResNetSmall (10–20× more parameters) with moderate regularization (weight_decay=1e-4) and a standard learning rate (0.001) for 20 epochs resolves the underfitting. The model now has sufficient capacity to learn the training distribution and enough epochs to converge.

**Key takeaway**: Model capacity must be matched to task complexity. Too little capacity or too much regularization prevents learning entirely.

## 8. Hyperparameter Tuning

We perform a systematic grid search over three key hyperparameters using ResNetSmall with CosineAnnealingLR and label smoothing:

| Hyperparameter | Values |
|---|---|
| Learning Rate | 0.01, 0.001, 0.0001 |
| Batch Size | 32, 64, 128 |
| Weight Decay | 0, 1e-4, 1e-3 |

Each combination is trained for 15 epochs (shorter to keep total runtime manageable).
We record the best validation accuracy for each combination.

In [ ]:
HP_EPOCHS = 15
lr_values = [0.01, 0.001, 0.0001]
bs_values = [32, 64, 128]
wd_values = [0, 1e-4, 1e-3]

hp_results = []

for lr in lr_values:
    for bs in bs_values:
        for wd in wd_values:
            set_global_seed(config.seed)
            cfg_hp = make_config_variant(
                architecture='resnet_small',
                epochs=HP_EPOCHS,
                early_stopping_patience=HP_EPOCHS + 1,
                learning_rate=lr,
                batch_size=bs,
                weight_decay=wd,
                loss_function='label_smoothing',
                label_smoothing=0.1,
                scheduler='cosine_annealing',
                scheduler_params={'T_max': HP_EPOCHS},
                weights_path=os.path.join(output_dir, f'hp_lr{lr}_bs{bs}_wd{wd}.pt'),
            )
            tl_hp, vl_hp = build_loaders(
                train_paths, train_labels, val_paths, val_labels, cfg_hp, mean, std
            )
            trainer_hp = Trainer(create_model(cfg_hp), tl_hp, vl_hp, cfg_hp)
            hist_hp = trainer_hp.train()
            best_val = max(hist_hp['val_acc'])
            hp_results.append({
                'lr': lr, 'batch_size': bs, 'weight_decay': wd,
                'best_val_acc': best_val,
            })
            print(f'LR={lr}, BS={bs}, WD={wd} → best val acc: {best_val:.4f}')

print(f'\nTotal combinations evaluated: {len(hp_results)}')

In [ ]:
# Display results table sorted by best val accuracy
hp_results_sorted = sorted(hp_results, key=lambda x: x['best_val_acc'], reverse=True)

print(f'{"Rank":>4s}  {"LR":>8s}  {"BS":>4s}  {"WD":>8s}  {"Val Acc":>8s}')
print('-' * 42)
for i, r in enumerate(hp_results_sorted, 1):
    print(f'{i:4d}  {r["lr"]:8.4f}  {r["batch_size"]:4d}  {r["weight_decay"]:8.5f}  {r["best_val_acc"]:8.4f}')

best_hp = hp_results_sorted[0]
print(f'\n✅ Best hyperparameters:')
print(f'   Learning Rate: {best_hp["lr"]}')
print(f'   Batch Size:    {best_hp["batch_size"]}')
print(f'   Weight Decay:  {best_hp["weight_decay"]}')
print(f'   Val Accuracy:  {best_hp["best_val_acc"]:.4f}')

### Hyperparameter Tuning Analysis

The grid search reveals several patterns:

- **Learning rate** is the most impactful hyperparameter. LR=0.01 often causes instability, while LR=0.0001 converges too slowly in 15 epochs. LR=0.001 consistently performs well.
- **Batch size** has a moderate effect. Smaller batches (32) provide noisier gradients that can help generalization, while larger batches (128) are more stable but may converge to sharper minima.
- **Weight decay** provides mild regularization. A small value (1e-4) typically helps, while too much (1e-3) can hurt performance.

We use the best combination from the grid search for the final training run.

## 9. Final Training

We train the final model using the best configuration identified above:
- **Architecture**: ResNetSmall
- **Loss**: Label Smoothing CrossEntropy (smoothing=0.1)
- **Scheduler**: CosineAnnealingLR
- **Hyperparameters**: Best LR, batch size, and weight decay from the grid search
- **Early stopping**: patience=10 on validation loss
- **Full epochs**: up to 100 (or until early stopping triggers)

The best model weights are saved to `outputs/best_model.pt`.

In [ ]:
# Build final config with best hyperparameters
set_global_seed(config.seed)

cfg_final = make_config_variant(
    architecture='resnet_small',
    epochs=config.epochs,  # full epochs (100)
    early_stopping_patience=config.early_stopping_patience,  # 10
    learning_rate=best_hp['lr'],
    batch_size=best_hp['batch_size'],
    weight_decay=best_hp['weight_decay'],
    loss_function='cross_entropy',
    label_smoothing=0.1,
    scheduler='cosine_annealing',
    scheduler_params={'T_max': config.epochs},
    weights_path=os.path.join(output_dir, 'best_model.pt'),
)

print('Final training configuration:')
print(f'  Architecture:  {cfg_final.architecture}')
print(f'  Learning Rate: {cfg_final.learning_rate}')
print(f'  Batch Size:    {cfg_final.batch_size}')
print(f'  Weight Decay:  {cfg_final.weight_decay}')
print(f'  Loss:          {cfg_final.loss_function} (smoothing={cfg_final.label_smoothing})')
print(f'  Scheduler:     {cfg_final.scheduler}')
print(f'  Max Epochs:    {cfg_final.epochs}')
print(f'  Early Stop:    patience={cfg_final.early_stopping_patience}')

In [ ]:
# Build loaders and train
tl_final, vl_final = build_loaders(
    train_paths, train_labels, val_paths, val_labels, cfg_final, mean, std
)
model_final = create_model(cfg_final)
print(f'Model params: {count_params(model_final):,}')

trainer_final = Trainer(model_final, tl_final, vl_final, cfg_final)
history_final = trainer_final.train()

In [ ]:
# Plot final training curves
plot_training_curves(history_final)
plot_lr_curve(history_final)

In [ ]:
# Summary
best_val_acc = max(history_final['val_acc'])
best_epoch = history_final['best_epoch']
stopped_epoch = history_final['stopped_epoch']

print(f'Best validation accuracy: {best_val_acc:.4f} (epoch {best_epoch})')
print(f'Training stopped at epoch: {stopped_epoch}')
print(f'Model weights saved to: {cfg_final.weights_path}')
print(f'\n--- Training complete. Ready for evaluation in notebook 03. ---')

## 10. Save Results to GitHub (Colab only)

In [ ]:
if IN_COLAB:
    from getpass import getpass
    TOKEN = getpass('GitHub Personal Access Token: ')
    
    !git config user.email "sabyasachi.kabiraj2@gmail.com"
    !git config user.name "Sabyasachi Kabiraj"
    
    !git add outputs/ notebooks/02_training.ipynb
    !git commit -m "Notebook 02: training results from Colab GPU"
    !git remote set-url origin https://{TOKEN}@github.com/sabyasachi1992/eurosat-land-cover-ood.git
    !git push origin main
    print('Pushed to GitHub.')
else:
    print('Not on Colab — skip git push.')